In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp -r /content/drive/MyDrive/AIdailyhub-projects/hockey_analytics/webapp.zip /content/.
!unzip /content/webapp.zip

Archive:  /content/webapp.zip
   creating: content/webapp/
   creating: content/webapp/models/
  inflating: content/webapp/models/D_reg.pkl  
  inflating: content/webapp/models/F_clf.pkl  
  inflating: content/webapp/models/cat_lookup.pkl  
  inflating: content/webapp/models/gl_clf.pkl  
  inflating: content/webapp/models/gl_reg.pkl  
  inflating: content/webapp/models/F_reg.pkl  
  inflating: content/webapp/models/goalie_features.pkl  
  inflating: content/webapp/models/D_clf.pkl  
  inflating: content/webapp/models/model_features_v3.pkl  
   creating: content/webapp/data/
  inflating: content/webapp/data/goalie_features.csv  
  inflating: content/webapp/data/skater_scored.csv  
  inflating: content/webapp/data/skater_features.csv  
  inflating: content/webapp/data/player_lookup.csv  
  inflating: content/webapp/data/goalie_scored.csv  


In [3]:
!mv /content/content/webapp /content/webapp
!rm -rf /content/content/

In [4]:
%cd /content/webapp
%ls

/content/webapp
data/  models/


In [6]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.9 MB/s eta 0:00:00


In [9]:
%%writefile app.py
"""
Hockey Trade Evaluator — Streamlit App
Uses pre-trained F/D/G models to score players and evaluate trades.
"""

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import plotly.graph_objects as go

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Hockey Trade Evaluator",
    page_icon="🏒",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─────────────────────────────────────────────
# STYLING
# ─────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Barlow+Condensed:wght@400;600;700&family=Barlow:wght@400;500&display=swap');

html, body, [class*="css"] {
    font-family: 'Barlow', sans-serif;
    background-color: #0a0e1a;
    color: #e0e6f0;
}

h1, h2, h3 {
    font-family: 'Barlow Condensed', sans-serif;
    letter-spacing: 0.04em;
}

/* Main header */
.main-header {
    background: linear-gradient(135deg, #0d1b2e 0%, #1a2a4a 100%);
    border-bottom: 2px solid #1e3a5f;
    padding: 1.2rem 2rem;
    margin: -1rem -1rem 1.5rem -1rem;
}
.main-header h1 {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 2rem;
    font-weight: 700;
    color: #ffffff;
    margin: 0;
    letter-spacing: 0.06em;
    text-transform: uppercase;
}
.main-header p {
    color: #7a9cc4;
    margin: 0.2rem 0 0 0;
    font-size: 0.85rem;
    letter-spacing: 0.05em;
}

/* Tab styling */
.stTabs [data-baseweb="tab-list"] {
    background-color: #0d1420;
    border-bottom: 2px solid #1e3a5f;
    gap: 0;
}
.stTabs [data-baseweb="tab"] {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 1rem;
    font-weight: 600;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    color: #5a7a9a;
    padding: 0.8rem 2rem;
    border-bottom: 3px solid transparent;
}
.stTabs [aria-selected="true"] {
    color: #4db8ff !important;
    border-bottom: 3px solid #4db8ff !important;
    background-color: transparent !important;
}

/* Player card */
.player-card {
    background: #111827;
    border: 1px solid #1e3a5f;
    border-radius: 8px;
    padding: 1rem 1.2rem;
    margin-bottom: 0.8rem;
    transition: border-color 0.2s;
}
.player-card:hover { border-color: #2a5a8f; }
.player-card-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 0.6rem;
}
.player-name {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 1.15rem;
    font-weight: 700;
    color: #ffffff;
    letter-spacing: 0.04em;
    text-transform: uppercase;
}
.player-pos {
    font-size: 0.75rem;
    font-weight: 600;
    padding: 0.2rem 0.6rem;
    border-radius: 4px;
    letter-spacing: 0.08em;
}
.pos-F  { background: #1a3a5c; color: #4db8ff; }
.pos-D  { background: #1a3c1a; color: #4dff88; }
.pos-G  { background: #3c2a1a; color: #ffb84d; }

.badge-green  { background:#0d3320; color:#34d399; border:1px solid #34d399;
                padding:0.2rem 0.7rem; border-radius:20px; font-size:0.78rem;
                font-weight:600; letter-spacing:0.06em; }
.badge-yellow { background:#2d2500; color:#fbbf24; border:1px solid #fbbf24;
                padding:0.2rem 0.7rem; border-radius:20px; font-size:0.78rem;
                font-weight:600; letter-spacing:0.06em; }
.badge-red    { background:#300d0d; color:#f87171; border:1px solid #f87171;
                padding:0.2rem 0.7rem; border-radius:20px; font-size:0.78rem;
                font-weight:600; letter-spacing:0.06em; }
.badge-warn   { background:#2d1a00; color:#fb923c; border:1px solid #fb923c;
                padding:0.2rem 0.7rem; border-radius:20px; font-size:0.72rem;
                font-weight:600; letter-spacing:0.05em; }

.stat-row {
    display: flex;
    gap: 1.5rem;
    flex-wrap: wrap;
    margin-top: 0.5rem;
}
.stat-item { text-align: center; }
.stat-label {
    font-size: 0.65rem;
    color: #5a7a9a;
    text-transform: uppercase;
    letter-spacing: 0.08em;
}
.stat-value {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 1.1rem;
    font-weight: 700;
    color: #e0e6f0;
}

/* Probability bar */
.prob-bar-wrap { margin-top: 0.6rem; }
.prob-label {
    font-size: 0.7rem;
    color: #5a7a9a;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    margin-bottom: 0.2rem;
}
.prob-bar-bg {
    background: #1e2d40;
    border-radius: 4px;
    height: 8px;
    width: 100%;
}
.prob-bar-fill {
    height: 8px;
    border-radius: 4px;
    transition: width 0.4s;
}

/* Team column */
.team-header {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 1.3rem;
    font-weight: 700;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    padding: 0.6rem 1rem;
    border-radius: 6px;
    margin-bottom: 1rem;
    text-align: center;
}
.team-a { background: #0d1e3a; color: #4db8ff; border: 1px solid #1e3a6e; }
.team-b { background: #1a0d0d; color: #f87171; border: 1px solid #6e1e1e; }

/* Summary verdict box */
.verdict-box {
    background: linear-gradient(135deg, #0d1b2e, #111827);
    border: 1px solid #1e3a5f;
    border-radius: 10px;
    padding: 1.5rem 2rem;
    text-align: center;
    margin: 1rem 0;
}
.verdict-title {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 0.85rem;
    color: #5a7a9a;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin-bottom: 0.4rem;
}
.verdict-main {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 2rem;
    font-weight: 700;
    letter-spacing: 0.04em;
}

/* Sidebar */
section[data-testid="stSidebar"] {
    background-color: #0d1420;
    border-right: 1px solid #1e3a5f;
}
section[data-testid="stSidebar"] .stSelectbox label,
section[data-testid="stSidebar"] .stMultiSelect label {
    color: #7a9cc4;
    font-size: 0.8rem;
    letter-spacing: 0.06em;
    text-transform: uppercase;
}

/* Divider */
.section-divider {
    border: none;
    border-top: 1px solid #1e3a5f;
    margin: 1.2rem 0;
}

/* Info metric box */
.metric-box {
    background: #111827;
    border: 1px solid #1e3a5f;
    border-radius: 8px;
    padding: 1rem;
    text-align: center;
}
.metric-box .metric-val {
    font-family: 'Barlow Condensed', sans-serif;
    font-size: 2rem;
    font-weight: 700;
    color: #4db8ff;
}
.metric-box .metric-lbl {
    font-size: 0.72rem;
    color: #5a7a9a;
    text-transform: uppercase;
    letter-spacing: 0.08em;
}
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# DATA & MODEL LOADING
# ─────────────────────────────────────────────
BASE = os.path.dirname(__file__)
DATA = os.path.join(BASE, "data")
MDLS = os.path.join(BASE, "models")

@st.cache_resource
def load_models():
    return {
        'F_clf':  joblib.load(f"{MDLS}/F_clf.pkl"),
        'F_reg':  joblib.load(f"{MDLS}/F_reg.pkl"),
        'D_clf':  joblib.load(f"{MDLS}/D_clf.pkl"),
        'D_reg':  joblib.load(f"{MDLS}/D_reg.pkl"),
        'gl_clf': joblib.load(f"{MDLS}/gl_clf.pkl"),
        'gl_reg': joblib.load(f"{MDLS}/gl_reg.pkl"),
        'sk_feats': joblib.load(f"{MDLS}/model_features_v3.pkl"),
        'gl_feats': joblib.load(f"{MDLS}/goalie_features.pkl"),
    }

@st.cache_data
def load_data():
    skaters = pd.read_csv(f"{DATA}/skater_scored.csv")
    goalies = pd.read_csv(f"{DATA}/goalie_scored.csv")
    lookup  = pd.read_csv(f"{DATA}/player_lookup.csv")
    return skaters, goalies, lookup

models          = load_models()
skater_df, goalie_df, player_lookup = load_data()

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
GOALIE_RULES = [
    ('age_at_trade',                 '<',  28.0),
    ('gl_plat_SHD_GAR',             '>',  -1.3),
    ('acquiring_pts_pct',            '<',   0.65),
    ('gl_plat_EVD_GAR',             '>',   1.0),
    ('gl_plat_TOI_SH',              '>',  130.0),
    ('total_pick_value_surrendered', '<',   5.0),
    ('gl_plat_GAR_per60',           '>',   0.23),
]

def zone_badge(is_green, is_red, is_goalie=False):
    warn = ' <span class="badge-warn">⚠️ Low conf</span>' if is_goalie else ''
    if is_green:
        return f'<span class="badge-green">🟢 Green Zone</span>{warn}'
    elif is_red:
        return f'<span class="badge-red">🔴 Red Zone</span>{warn}'
    else:
        return f'<span class="badge-yellow">🟡 Middle</span>{warn}'

def prob_bar(prob, color="#4db8ff"):
    pct = int(prob * 100)
    return f"""
    <div class="prob-bar-wrap">
      <div class="prob-label">Beat-Market Probability</div>
      <div style="display:flex;align-items:center;gap:0.6rem;">
        <div class="prob-bar-bg" style="flex:1">
          <div class="prob-bar-fill" style="width:{pct}%;background:{color}"></div>
        </div>
        <span style="font-family:'Barlow Condensed',sans-serif;font-weight:700;
                     color:{color};font-size:1.1rem;min-width:3rem">{pct}%</span>
      </div>
    </div>"""

def get_player_row(name, df):
    rows = df[df['eh_name'] == name]
    if len(rows) == 0:
        return None
    return rows.sort_values('trade_season', ascending=False).iloc[0]

def score_player(row, pos):
    """Run model prediction for a player row."""
    if pos in ('F', 'D'):
        feats = models['sk_feats']
        # Rebuild engineered features if needed
        row = row.copy()
        age = row.get('age_at_trade', 27)
        row['age_squared']   = age ** 2
        row['age_cubed']     = age ** 3
        row['age_peak_dist'] = abs(age - 27)
        row['past_peak_flag']= int(age > 27)
        X = np.array([[row.get(f, 0) for f in feats]])
        clf = models[f'{pos}_clf']
        reg = models[f'{pos}_reg']
        prob    = clf.predict_proba(X)[0, 1]
        surplus = reg.predict(X)[0]
        is_green = int(prob >= 0.60)
        is_red   = int(prob <  0.40)
        rules_met = None
    else:
        feats = models['gl_feats']
        X = np.array([[row.get(f, 0) for f in feats]])
        prob    = models['gl_clf'].predict_proba(X)[0, 1]
        surplus = models['gl_reg'].predict(X)[0]
        rules_met = sum(
            1 for feat, op, thresh in GOALIE_RULES
            if (op == '<' and row.get(feat, 0) < thresh) or
               (op == '>' and row.get(feat, 0) > thresh)
        )
        is_green = int(rules_met >= 5)
        is_red   = int(rules_met <= 2)
    return prob, surplus, is_green, is_red, rules_met

def render_player_card(name, pos, row, team_color="#4db8ff"):
    prob, surplus, is_green, is_red, rules_met = score_player(row, pos)
    is_goalie = (pos == 'G')

    bar_color    = "#34d399" if is_green else ("#f87171" if is_red else "#fbbf24")
    surplus_str  = f"{surplus:+.1f} GAR"
    surplus_color = "#34d399" if surplus > 0 else "#f87171"
    pct           = int(prob * 100)

    # Position badge colours
    pos_bg  = {"F": "#1a3a5c", "D": "#1a3c1a", "G": "#3c2a1a"}
    pos_fg  = {"F": "#4db8ff", "D": "#4dff88", "G": "#ffb84d"}

    # Zone badge
    if is_green:
        zone_bg, zone_fg, zone_bd, zone_txt = "#0d3320","#34d399","#34d399","🟢 Green Zone"
    elif is_red:
        zone_bg, zone_fg, zone_bd, zone_txt = "#300d0d","#f87171","#f87171","🔴 Red Zone"
    else:
        zone_bg, zone_fg, zone_bd, zone_txt = "#2d2500","#fbbf24","#fbbf24","🟡 Middle"

    goalie_warn = (
        '<span style="background:#2d1a00;color:#fb923c;border:1px solid #fb923c;'
        'padding:0.15rem 0.5rem;border-radius:20px;font-size:0.7rem;'
        'font-weight:600;letter-spacing:0.05em;margin-left:0.4rem;">⚠️ Low conf</span>'
        if is_goalie else ""
    )

    # Stat items
    STAT = ('<div style="text-align:center;">'
            '<div style="font-size:0.65rem;color:#5a7a9a;text-transform:uppercase;'
            'letter-spacing:0.08em;">{label}</div>'
            '<div style="font-family:\'Barlow Condensed\',sans-serif;font-size:1.1rem;'
            'font-weight:700;color:#e0e6f0;">{val}</div></div>')

    if pos == 'G':
        stats_html = "".join([
            STAT.format(label="GAR",      val=f"{row.get('gl_plat_GAR', 0):.1f}"),
            STAT.format(label="SHD GAR",  val=f"{row.get('gl_plat_SHD_GAR', 0):.1f}"),
            STAT.format(label="EVD GAR",  val=f"{row.get('gl_plat_EVD_GAR', 0):.1f}"),
            STAT.format(label="GP",       val=f"{row.get('gl_plat_GP', 0):.0f}"),
            STAT.format(label="Age",      val=f"{row.get('age_at_trade', 0):.1f}"),
            STAT.format(label="Rules Met",val=f"{rules_met}/7"),
        ])
    else:
        stats_html = "".join([
            STAT.format(label="GAR",     val=f"{row.get('platform_GAR', 0):.1f}"),
            STAT.format(label="GAR/60",  val=f"{row.get('platform_GAR_per60', 0):.2f}"),
            STAT.format(label="Pts/60",  val=f"{row.get('plat_points_per60', 0):.2f}"),
            STAT.format(label="ATOI",    val=f"{row.get('plat_atoi', 0):.0f} min"),
            STAT.format(label="Age",     val=f"{row.get('age_at_trade', 0):.1f}"),
            STAT.format(label="Cap%",    val=f"{row.get('aav_pct_cap', 0)*100:.1f}%"),
        ])

    goalie_reg_note = (
        '<span style="color:#fb923c;font-size:0.7rem;margin-left:0.5rem;">'
        '⚠️ reg unreliable</span>' if is_goalie else ""
    )

    st.markdown(f"""
    <div style="background:#111827;border:1px solid #1e3a5f;border-radius:8px;
                padding:1rem 1.2rem;margin-bottom:0.8rem;">

      <!-- Header row -->
      <div style="display:flex;justify-content:space-between;
                  align-items:center;margin-bottom:0.7rem;">
        <div style="display:flex;align-items:center;gap:0.5rem;">
          <span style="font-family:'Barlow Condensed',sans-serif;font-size:1.15rem;
                       font-weight:700;color:#ffffff;letter-spacing:0.04em;
                       text-transform:uppercase;">{name}</span>
          <span style="background:{pos_bg[pos]};color:{pos_fg[pos]};
                       font-size:0.75rem;font-weight:600;padding:0.2rem 0.6rem;
                       border-radius:4px;letter-spacing:0.08em;">{pos}</span>
        </div>
        <div style="display:flex;align-items:center;gap:0.3rem;">
          <span style="background:{zone_bg};color:{zone_fg};border:1px solid {zone_bd};
                       padding:0.2rem 0.7rem;border-radius:20px;font-size:0.78rem;
                       font-weight:600;letter-spacing:0.06em;">{zone_txt}</span>
          {goalie_warn}
        </div>
      </div>

      <!-- Stats row -->
      <div style="display:flex;gap:1.2rem;flex-wrap:wrap;margin-bottom:0.7rem;">
        {stats_html}
      </div>

      <!-- Probability bar -->
      <div style="margin-bottom:0.5rem;">
        <div style="font-size:0.68rem;color:#5a7a9a;text-transform:uppercase;
                    letter-spacing:0.08em;margin-bottom:0.25rem;">
          Beat-Market Probability
        </div>
        <div style="display:flex;align-items:center;gap:0.6rem;">
          <div style="flex:1;background:#1e2d40;border-radius:4px;height:8px;">
            <div style="width:{pct}%;background:{bar_color};
                        height:8px;border-radius:4px;"></div>
          </div>
          <span style="font-family:'Barlow Condensed',sans-serif;font-weight:700;
                       color:{bar_color};font-size:1.1rem;min-width:3rem;">{pct}%</span>
        </div>
      </div>

      <!-- Surplus -->
      <div style="font-size:0.78rem;color:#5a7a9a;">
        Predicted surplus:
        <span style="color:{surplus_color};font-weight:700;
                     font-family:'Barlow Condensed',sans-serif;">
          {surplus_str}
        </span>
        {goalie_reg_note}
      </div>

    </div>
    """, unsafe_allow_html=True)

    return prob, surplus, is_green, is_red

# ─────────────────────────────────────────────
# SESSION STATE
# ─────────────────────────────────────────────
if 'team_a_players' not in st.session_state:
    st.session_state.team_a_players = []
if 'team_b_players' not in st.session_state:
    st.session_state.team_b_players = []
if 'team_a_name' not in st.session_state:
    st.session_state.team_a_name = "Team A"
if 'team_b_name' not in st.session_state:
    st.session_state.team_b_name = "Team B"

all_players = sorted(player_lookup['eh_name'].unique().tolist())

# ─────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────
st.markdown("""
<div class="main-header">
  <h1>🏒 Hockey Trade Evaluator</h1>
  <p>NHL TRANSACTION SURPLUS MODEL &nbsp;·&nbsp;
     FORWARDS · DEFENSEMEN · GOALIES</p>
</div>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
with st.sidebar:
    st.markdown("### 🏒 Trade Setup")
    st.markdown("<hr class='section-divider'>", unsafe_allow_html=True)

    st.session_state.team_a_name = st.text_input(
        "Team A Name", value=st.session_state.team_a_name)
    team_a_picks = st.number_input(
        "Team A pick value surrendered", min_value=0.0,
        max_value=100.0, value=0.0, step=0.5,
        help="Total draft pick value Team A gives away")

    st.markdown("<hr class='section-divider'>", unsafe_allow_html=True)

    st.session_state.team_b_name = st.text_input(
        "Team B Name", value=st.session_state.team_b_name)
    team_b_picks = st.number_input(
        "Team B pick value surrendered", min_value=0.0,
        max_value=100.0, value=0.0, step=0.5,
        help="Total draft pick value Team B gives away")

    st.markdown("<hr class='section-divider'>", unsafe_allow_html=True)
    st.markdown("**Model Info**")
    st.caption(f"Players in database: {len(all_players):,}")
    st.caption(f"Skater features: {len(models['sk_feats'])}")
    st.caption(f"Goalie features: {len(models['gl_feats'])}")
    # st.caption("⚠️ Goalie regression low confidence")

# ─────────────────────────────────────────────
# TABS
# ─────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs([
    "🔧  Trade Builder",
    "📋  Player Scorecards",
    "📊  Trade Summary"
])

# ═════════════════════════════════════════════
# TAB 1: TRADE BUILDER
# ═════════════════════════════════════════════
with tab1:
    col_a, col_b = st.columns(2)

    # ── Team A ────────────────────────────────
    with col_a:
        st.markdown(
            f'<div class="team-header team-a">'
            f'{st.session_state.team_a_name} Gives</div>',
            unsafe_allow_html=True)

        new_a = st.selectbox(
            "Add player to Team A",
            options=["— select player —"] + all_players,
            key="add_a")
        if st.button("➕ Add to Trade", key="btn_a"):
            if new_a != "— select player —" and \
               new_a not in st.session_state.team_a_players:
                st.session_state.team_a_players.append(new_a)

        # Show current roster
        if st.session_state.team_a_players:
            st.markdown("**Players moving from Team A:**")
            for p in list(st.session_state.team_a_players):
                row = player_lookup[player_lookup['eh_name'] == p]
                pos = row['position_group'].values[0] if len(row) else "?"
                c1, c2 = st.columns([4, 1])
                with c1:
                    st.markdown(f"`{pos}` &nbsp; {p}", unsafe_allow_html=True)
                with c2:
                    if st.button("✕", key=f"rm_a_{p}"):
                        st.session_state.team_a_players.remove(p)
                        st.rerun()
        else:
            st.info("No players added yet.")

        if team_a_picks > 0:
            st.markdown(
                f"🎯 **Pick value surrendered:** `{team_a_picks:.1f} pts`")

    # ── Team B ────────────────────────────────
    with col_b:
        st.markdown(
            f'<div class="team-header team-b">'
            f'{st.session_state.team_b_name} Gives</div>',
            unsafe_allow_html=True)

        new_b = st.selectbox(
            "Add player to Team B",
            options=["— select player —"] + all_players,
            key="add_b")
        if st.button("➕ Add to Trade", key="btn_b"):
            if new_b != "— select player —" and \
               new_b not in st.session_state.team_b_players:
                st.session_state.team_b_players.append(new_b)

        if st.session_state.team_b_players:
            st.markdown("**Players moving from Team B:**")
            for p in list(st.session_state.team_b_players):
                row = player_lookup[player_lookup['eh_name'] == p]
                pos = row['position_group'].values[0] if len(row) else "?"
                c1, c2 = st.columns([4, 1])
                with c1:
                    st.markdown(f"`{pos}` &nbsp; {p}", unsafe_allow_html=True)
                with c2:
                    if st.button("✕", key=f"rm_b_{p}"):
                        st.session_state.team_b_players.remove(p)
                        st.rerun()
        else:
            st.info("No players added yet.")

        if team_b_picks > 0:
            st.markdown(
                f"🎯 **Pick value surrendered:** `{team_b_picks:.1f} pts`")

    st.markdown("<hr class='section-divider'>", unsafe_allow_html=True)

    # Clear button
    if st.button("🗑️  Clear All Players"):
        st.session_state.team_a_players = []
        st.session_state.team_b_players = []
        st.rerun()

    total = (len(st.session_state.team_a_players) +
             len(st.session_state.team_b_players))
    if total > 0:
        st.success(
            f"✅ {total} player(s) added. "
            f"Go to **Player Scorecards** or **Trade Summary** tabs.")
    else:
        st.markdown("""
        **How to use:**
        1. Enter team names in the sidebar
        2. Select players each team gives away
        3. Add pick values if applicable
        4. Check scorecards and summary tabs
        """)

# ═════════════════════════════════════════════
# TAB 2: PLAYER SCORECARDS
# ═════════════════════════════════════════════
with tab2:
    all_trade_players = (
        [(p, 'A') for p in st.session_state.team_a_players] +
        [(p, 'B') for p in st.session_state.team_b_players]
    )

    if not all_trade_players:
        st.info("Add players in the Trade Builder tab to see scorecards.")
    else:
        col_a2, col_b2 = st.columns(2)

        team_a_results = []
        team_b_results = []

        with col_a2:
            st.markdown(
                f'<div class="team-header team-a">'
                f'{st.session_state.team_a_name} Receives</div>',
                unsafe_allow_html=True)
            for p in st.session_state.team_b_players:
                lkp = player_lookup[player_lookup['eh_name'] == p]
                if len(lkp) == 0:
                    st.warning(f"{p} not found in database.")
                    continue
                pos = lkp['position_group'].values[0]
                src = goalie_df if pos == 'G' else skater_df
                row = get_player_row(p, src)
                if row is None:
                    st.warning(f"No data found for {p}.")
                    continue
                prob, surplus, is_green, is_red = render_player_card(
                    p, pos, row, "#4db8ff")
                team_a_results.append({
                    'name': p, 'pos': pos,
                    'prob': prob, 'surplus': surplus,
                    'is_green': is_green, 'is_red': is_red
                })

        with col_b2:
            st.markdown(
                f'<div class="team-header team-b">'
                f'{st.session_state.team_b_name} Receives</div>',
                unsafe_allow_html=True)
            for p in st.session_state.team_a_players:
                lkp = player_lookup[player_lookup['eh_name'] == p]
                if len(lkp) == 0:
                    st.warning(f"{p} not found in database.")
                    continue
                pos = lkp['position_group'].values[0]
                src = goalie_df if pos == 'G' else skater_df
                row = get_player_row(p, src)
                if row is None:
                    st.warning(f"No data found for {p}.")
                    continue
                prob, surplus, is_green, is_red = render_player_card(
                    p, pos, row, "#f87171")
                team_b_results.append({
                    'name': p, 'pos': pos,
                    'prob': prob, 'surplus': surplus,
                    'is_green': is_green, 'is_red': is_red
                })

        # Store results in session for summary tab
        st.session_state['team_a_results'] = team_a_results
        st.session_state['team_b_results'] = team_b_results

# ═════════════════════════════════════════════
# TAB 3: TRADE SUMMARY
# ═════════════════════════════════════════════
with tab3:
    team_a_res = st.session_state.get('team_a_results', [])
    team_b_res = st.session_state.get('team_b_results', [])

    if not team_a_res and not team_b_res:
        st.info(
            "Add players in Trade Builder and view Scorecards "
            "first to generate the summary.")
    else:
        # ── Aggregate scores ──────────────────────────────────────
        def agg(results, pick_penalty=0.0):
            if not results:
                return 0.0, 0.0, 0, 0
            avg_prob    = np.mean([r['prob']    for r in results])
            total_surp  = sum([r['surplus'] for r in results]) - pick_penalty
            n_green     = sum([r['is_green'] for r in results])
            n_red       = sum([r['is_red']   for r in results])
            return avg_prob, total_surp, n_green, n_red

        a_prob, a_surp, a_green, a_red = agg(team_a_res, team_a_picks * 0.5)
        b_prob, b_surp, b_green, b_red = agg(team_b_res, team_b_picks * 0.5)

        # ── Verdict ───────────────────────────────────────────────
        has_goalie_a = any(r['pos'] == 'G' for r in team_a_res)
        has_goalie_b = any(r['pos'] == 'G' for r in team_b_res)
        has_goalie   = has_goalie_a or has_goalie_b
        conf_note    = " (moderate conf — goalie in deal)" if has_goalie else ""

        if abs(a_surp - b_surp) < 1.0:
            verdict      = "⚖️  EVEN TRADE"
            verdict_col  = "#fbbf24"
        elif a_surp > b_surp:
            verdict     = f"🏆  {st.session_state.team_a_name} WINS"
            verdict_col = "#34d399"
        else:
            verdict     = f"🏆  {st.session_state.team_b_name} WINS"
            verdict_col = "#34d399"

        st.markdown(f"""
        <div class="verdict-box">
          <div class="verdict-title">Trade Verdict{conf_note}</div>
          <div class="verdict-main" style="color:{verdict_col}">{verdict}</div>
        </div>
        """, unsafe_allow_html=True)

        # ── Metrics row ───────────────────────────────────────────
        m1, m2, m3, m4 = st.columns(4)
        with m1:
            st.markdown(f"""
            <div class="metric-box">
              <div class="metric-val" style="color:#4db8ff">
                {a_prob*100:.0f}%
              </div>
              <div class="metric-lbl">
                {st.session_state.team_a_name} avg beat-mkt
              </div>
            </div>""", unsafe_allow_html=True)
        with m2:
            sc = "#34d399" if a_surp >= 0 else "#f87171"
            st.markdown(f"""
            <div class="metric-box">
              <div class="metric-val" style="color:{sc}">
                {a_surp:+.1f}
              </div>
              <div class="metric-lbl">
                {st.session_state.team_a_name} net surplus GAR
              </div>
            </div>""", unsafe_allow_html=True)
        with m3:
            st.markdown(f"""
            <div class="metric-box">
              <div class="metric-val" style="color:#f87171">
                {b_prob*100:.0f}%
              </div>
              <div class="metric-lbl">
                {st.session_state.team_b_name} avg beat-mkt
              </div>
            </div>""", unsafe_allow_html=True)
        with m4:
            sc = "#34d399" if b_surp >= 0 else "#f87171"
            st.markdown(f"""
            <div class="metric-box">
              <div class="metric-val" style="color:{sc}">
                {b_surp:+.1f}
              </div>
              <div class="metric-lbl">
                {st.session_state.team_b_name} net surplus GAR
              </div>
            </div>""", unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)

        # ── Surplus bar chart ─────────────────────────────────────
        fig = go.Figure()

        names_a = [r['name'] for r in team_a_res]
        surps_a = [r['surplus'] for r in team_a_res]
        names_b = [r['name'] for r in team_b_res]
        surps_b = [r['surplus'] for r in team_b_res]

        all_names = names_a + names_b
        all_surps = surps_a + surps_b
        all_teams = (
            [st.session_state.team_a_name] * len(names_a) +
            [st.session_state.team_b_name] * len(names_b)
        )
        colors = (["#4db8ff"] * len(names_a) +
                  ["#f87171"] * len(names_b))
        bar_colors = ["#34d399" if s > 0 else "#f87171"
                      for s in all_surps]

        fig.add_trace(go.Bar(
            x=all_names,
            y=all_surps,
            marker_color=bar_colors,
            marker_line_color=colors,
            marker_line_width=2,
            text=[f"{s:+.1f}" for s in all_surps],
            textposition='outside',
            textfont=dict(color='white', size=12),
        ))

        fig.add_hline(y=0, line_color="#5a7a9a",
                      line_width=1, line_dash="dash")

        fig.update_layout(
            title=dict(
                text="Predicted Surplus by Player (GAR)",
                font=dict(color='white', size=14,
                          family='Barlow Condensed'),
            ),
            paper_bgcolor='#111827',
            plot_bgcolor='#111827',
            font=dict(color='white', family='Barlow'),
            xaxis=dict(
                tickfont=dict(color='white', size=11),
                gridcolor='#1e3a5f',
                showgrid=False,
            ),
            yaxis=dict(
                title="Predicted Surplus (GAR)",
                tickfont=dict(color='white'),
                gridcolor='#1e3a5f',
            ),
            margin=dict(t=50, b=20, l=20, r=20),
            height=350,
            showlegend=False,
        )
        st.plotly_chart(fig, use_container_width=True)

        # ── Beat-market prob comparison ────────────────────────────
        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            name=st.session_state.team_a_name,
            x=[r['name'] for r in team_a_res],
            y=[r['prob'] * 100 for r in team_a_res],
            marker_color='#4db8ff',
            text=[f"{r['prob']*100:.0f}%" for r in team_a_res],
            textposition='outside',
            textfont=dict(color='white'),
        ))
        fig2.add_trace(go.Bar(
            name=st.session_state.team_b_name,
            x=[r['name'] for r in team_b_res],
            y=[r['prob'] * 100 for r in team_b_res],
            marker_color='#f87171',
            text=[f"{r['prob']*100:.0f}%" for r in team_b_res],
            textposition='outside',
            textfont=dict(color='white'),
        ))
        fig2.add_hline(y=50, line_color="#5a7a9a",
                       line_width=1, line_dash="dash",
                       annotation_text="50% baseline",
                       annotation_font_color="#5a7a9a")
        fig2.update_layout(
            title=dict(
                text="Beat-Market Probability by Player (%)",
                font=dict(color='white', size=14,
                          family='Barlow Condensed'),
            ),
            paper_bgcolor='#111827',
            plot_bgcolor='#111827',
            font=dict(color='white', family='Barlow'),
            xaxis=dict(tickfont=dict(color='white', size=11),
                       showgrid=False),
            yaxis=dict(title="Probability (%)",
                       tickfont=dict(color='white'),
                       gridcolor='#1e3a5f',
                       range=[0, 110]),
            barmode='group',
            margin=dict(t=50, b=20, l=20, r=20),
            height=350,
            legend=dict(font=dict(color='white'),
                        bgcolor='#111827'),
        )
        st.plotly_chart(fig2, use_container_width=True)

        # ── Zone summary table ────────────────────────────────────
        st.markdown("<hr class='section-divider'>", unsafe_allow_html=True)
        st.markdown("##### Player Zone Summary")

        all_results = (
            [(r, st.session_state.team_a_name, 'Receives')
             for r in team_a_res] +
            [(r, st.session_state.team_b_name, 'Receives')
             for r in team_b_res]
        )
        summary_rows = []
        for r, team, direction in all_results:
            zone = ("🟢 Green" if r['is_green']
                    else ("🔴 Red" if r['is_red'] else "🟡 Middle"))
            goalie_note = " ⚠️" if r['pos'] == 'G' else ""
            summary_rows.append({
                'Team': team,
                'Player': r['name'],
                'Pos': r['pos'] + goalie_note,
                'Zone': zone,
                'Beat-Mkt Prob': f"{r['prob']*100:.0f}%",
                'Pred Surplus': f"{r['surplus']:+.1f} GAR",
            })

        if summary_rows:
            st.dataframe(
                pd.DataFrame(summary_rows),
                use_container_width=True,
                hide_index=True,
            )

        # ── Caveats ───────────────────────────────────────────────
        st.markdown("<hr class='section-divider'>", unsafe_allow_html=True)
        # st.caption(
        #     "⚠️ Goalie regression surplus is low confidence (LOSO R²=−0.074). "
        #     "Use beat-market probability as the primary goalie signal.  "
        #     "Skater models: F holdout R²=0.088 AUC=0.651 · "
        #     "D holdout R²=0.220 AUC=0.756.  "
        #     "All predictions based on platform season stats at time of trade."
        # )

Overwriting app.py


In [10]:
from pyngrok import ngrok

ngrok_key = '3BtL9LKsP10jXjQC460c7lfvepB_6iYXWn2kvHACbpoFM6A65'
port = 8501

ngrok.set_auth_token(ngrok_key)
ngrok.connect(port).public_url

'https://marcos-isacoustic-exhortingly.ngrok-free.dev'

In [11]:
!rm -rf logs.txt && streamlit run app.py &>/content/logs.txt